In [7]:

import os
import csv
import io
import getpass
import psycopg2

# ==================================================
# STEP 1: PostgreSQL Connection Settings
# ==================================================

db_password = getpass.getpass("Enter PostgreSQL password: ")

db_config = {
    "host": "localhost",
    "port": "5432",
    "dbname": "Health_DataBase",
    "user": "postgres",
    "password": db_password
}

# ==================================================
# STEP 2: Project Folder Location
# ==================================================

project_dir = (
    r"C:\Users\User\OneDrive\Desktop"
    r"\Health Analytics\Health_Analytics_sql-main"
)

clean_dir = os.path.join(project_dir, "data", "outputs")

# ==================================================
# STEP 3: Map CSV Files to PostgreSQL Tables
# ==================================================

file_to_table = {
    "cleaned_DimPatient.csv": "dimpatient",
    "cleaned_DimPhysician.csv": "dimphysician",
    "cleaned_DimSpeciality.csv": "dimspeciality",
    "cleaned_DimHospital.csv": "dimhospital",
    "cleaned_DimPayer.csv": "dimpayer",
    "cleaned_DimCptCode.csv": "dimcptcode",
    "cleaned_DimDiagnosisCode.csv": "dimdiagnosiscode",
    "cleaned_DimDate.csv": "dimdate",
    "cleaned_DimTransaction.csv": "dimtransaction",
    "cleaned_FactTable.csv": "facttable"
}

# ==================================================
# STEP 4: Verify Cleaned CSV Files
# ==================================================

print("\nChecking cleaned datasets...\n")

if not os.path.isdir(clean_dir):
    raise FileNotFoundError(
        f"Cleaned data folder not found: {clean_dir}"
    )

for filename in file_to_table:

    file_path = os.path.join(clean_dir, filename)

    if not os.path.isfile(file_path):
        raise FileNotFoundError(
            f"Missing CSV file: {file_path}"
        )

    print(f"Found: {filename}")

print("\nAll 10 cleaned CSV files found.")

# ==================================================
# STEP 5: Connect to PostgreSQL
# ==================================================

conn = None

try:

    conn = psycopg2.connect(**db_config)

    print("\nConnected to PostgreSQL successfully!")

    with conn.cursor() as cur:

        # ==========================================
        # STEP 6: Check Tables Are Empty
        # ==========================================

        print("\nChecking database tables...")

        for table in file_to_table.values():

            cur.execute(f"SELECT COUNT(*) FROM {table}")

            existing_rows = cur.fetchone()[0]

            if existing_rows > 0:
                raise RuntimeError(
                    f"Table '{table}' already contains "
                    f"{existing_rows} rows. "
                    "Import stopped to prevent duplicates."
                )

        print("All database tables are empty.")

        # ==========================================
        # STEP 7: Import CSV Files
        # ==========================================

        print("\nStarting data import...\n")

        for filename, table in file_to_table.items():

            file_path = os.path.join(clean_dir, filename)

            print(f"Loading {filename} into {table}...")

            # Read CSV and standardise missing values
            with open(
                file_path,
                "r",
                encoding="utf-8-sig",
                newline=""
            ) as f:

                reader = csv.reader(f)
                header = next(reader)

                # Check PostgreSQL column count
                cur.execute(
                    """
                    SELECT COUNT(*)
                    FROM information_schema.columns
                    WHERE table_schema = 'public'
                      AND table_name = %s
                    """,
                    (table,)
                )

                expected_columns = cur.fetchone()[0]

                if len(header) != expected_columns:
                    raise ValueError(
                        f"Column mismatch for {filename}: "
                        f"CSV has {len(header)} columns, "
                        f"but {table} has {expected_columns}."
                    )

                # Create an in-memory CSV
                # without modifying original files
                csv_buffer = io.StringIO()

                writer = csv.writer(
                    csv_buffer,
                    lineterminator="\n"
                )

                # Preserve the header
                writer.writerow(header)

                # Standardise missing values to NA
                for row in reader:

                    cleaned_row = [
                        "NA"
                        if value.strip() in ("", "NA", "nan", "NaN")
                        else value
                        for value in row
                    ]

                    writer.writerow(cleaned_row)

            # Move to beginning of in-memory CSV
            csv_buffer.seek(0)

            # Import into PostgreSQL
            # NA values become SQL NULL
            cur.copy_expert(
                f"""
                COPY {table}
                FROM STDIN
                WITH (
                    FORMAT CSV,
                    HEADER TRUE,
                    DELIMITER ',',
                    NULL 'NA'
                )
                """,
                csv_buffer
            )

            csv_buffer.close()

            # Verify imported row count
            cur.execute(f"SELECT COUNT(*) FROM {table}")

            loaded_rows = cur.fetchone()[0]

            print(
                f"Successfully loaded {loaded_rows:,} "
                f"rows into {table}\n"
            )

    # ==============================================
    # STEP 8: Commit All Imports
    # ==============================================

    conn.commit()

    print("=" * 50)
    print("SUCCESS: All 10 datasets imported!")
    print("=" * 50)

except Exception as e:

    if conn is not None:
        conn.rollback()

    print("\nERROR: Data import failed.")
    print(e)
    print("Changes rolled back. No partial import saved.")

finally:

    if conn is not None:
        conn.close()
        print("\nPostgreSQL connection closed.")


Enter PostgreSQL password:  ········



Checking cleaned datasets...

Found: cleaned_DimPatient.csv
Found: cleaned_DimPhysician.csv
Found: cleaned_DimSpeciality.csv
Found: cleaned_DimHospital.csv
Found: cleaned_DimPayer.csv
Found: cleaned_DimCptCode.csv
Found: cleaned_DimDiagnosisCode.csv
Found: cleaned_DimDate.csv
Found: cleaned_DimTransaction.csv
Found: cleaned_FactTable.csv

All 10 cleaned CSV files found.

Connected to PostgreSQL successfully!

Checking database tables...
All database tables are empty.

Starting data import...

Loading cleaned_DimPatient.csv into dimpatient...
Successfully loaded 5,117 rows into dimpatient

Loading cleaned_DimPhysician.csv into dimphysician...
Successfully loaded 932 rows into dimphysician

Loading cleaned_DimSpeciality.csv into dimspeciality...
Successfully loaded 34 rows into dimspeciality

Loading cleaned_DimHospital.csv into dimhospital...
Successfully loaded 11 rows into dimhospital

Loading cleaned_DimPayer.csv into dimpayer...
Successfully loaded 4 rows into dimpayer

Loading cle